In [1]:
import re
import string
import pandas as pd
import numpy as np
import nltk
from google.colab import files

# NLTK preprocessing tools
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Scikit-Learn TF-IDF, models, and evaluation metrics
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Download required NLTK datasets
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("Libraries and NLTK resources loaded successfully!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Libraries and NLTK resources loaded successfully!


In [2]:
# Upload the CSV file from your computer
uploaded = files.upload()

# Automatically grab the uploaded filename
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f"\nLoaded '{filename}' with shape: {df.shape}")
print("Columns found:", df.columns.tolist())
display(df.head())

# Check for missing values and drop them if present
print("\nMissing values per column:\n", df.isnull().sum())
df.dropna(inplace=True)

Saving sentiment_reviews.csv to sentiment_reviews.csv

Loaded 'sentiment_reviews.csv' with shape: (10, 3)
Columns found: ['review_id', 'review_text', 'sentiment']


,review_id,review_text,sentiment
0,1,"The product quality is exceptional, exceeded a...",positive
1,2,Terrible customer service and the item arrived...,negative
2,3,"Average performance, decent for the price poin...",neutral
3,4,"Absolutely love this! Easy to setup, robust, a...",positive
4,5,Stopped working after two days. Extremely disa...,negative



Missing values per column:
 review_id      0
review_text    0
sentiment      0
dtype: int64


In [5]:
# 1. Inspect actual columns and automatically pick the text column (longest average string length)
print("Available columns:", df.columns.tolist())

# Exclude obvious label/id names when searching for the review text column
non_text_keywords = ['id', 'sentiment', 'label', 'target', 'polarity', 'rating', 'class', 'score']
candidate_text_cols = [
    col for col in df.columns
    if df[col].dtype == 'object' and not any(k == col.lower() for k in non_text_keywords)
]

if candidate_text_cols:
    text_col = max(candidate_text_cols, key=lambda c: df[c].astype(str).str.len().mean())
else:
    text_col = max(df.columns, key=lambda c: df[c].astype(str).str.len().mean())

# Pick the target sentiment column (any remaining column, prioritizing label names)
label_candidates = ['sentiment', 'label', 'target', 'polarity', 'rating', 'class']
remaining_cols = [c for c in df.columns if c != text_col]
label_col = next((c for c in remaining_cols if c.lower() in label_candidates), remaining_cols[-1])

print(f"Selected Text Column:   '{text_col}'")
print(f"Selected Target Column: '{label_col}'\n")

# 2. Clean text safely without wiping out short reviews
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text)
    text = re.sub(r'<.*?>', ' ', text)                  # Remove HTML tags
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)              # Keep only letters
    text = text.lower()
    tokens = word_tokenize(text)

    cleaned_tokens = [
        lemmatizer.lemmatize(word) for word in tokens if word not in stop_words
    ]
    # Fallback: if all words were stopwords, keep original lowercase alphabetic tokens
    if not cleaned_tokens:
        cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(cleaned_tokens)

df['cleaned_review'] = df[text_col].apply(clean_text)

# Drop any rows that are still completely empty after cleaning
df = df[df['cleaned_review'].str.strip() != ""].reset_index(drop=True)

print(f"Total valid rows ready for TF-IDF: {len(df)}")
display(df[[text_col, 'cleaned_review', label_col]].head())

Available columns: ['review_id', 'review_text', 'sentiment', 'cleaned_review']
Selected Text Column:   'review_text'
Selected Target Column: 'sentiment'

Total valid rows ready for TF-IDF: 10


,review_text,cleaned_review,sentiment
0,"The product quality is exceptional, exceeded a...",product quality exceptional exceeded expectation,positive
1,Terrible customer service and the item arrived...,terrible customer service item arrived complet...,negative
2,"Average performance, decent for the price poin...",average performance decent price point nothing...,neutral
3,"Absolutely love this! Easy to setup, robust, a...",absolutely love easy setup robust work flawlessly,positive
4,Stopped working after two days. Extremely disa...,stopped working two day extremely disappointed...,negative


In [6]:
X = df['cleaned_review']
y = df[label_col]

try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
except ValueError:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

# Initialize and fit TfidfVectorizer with max_features=5000
tfidf = TfidfVectorizer(max_features=5000, token_pattern=r'\b\w+\b')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF Training Matrix Shape: {X_train_tfidf.shape}")
print(f"TF-IDF Testing Matrix Shape:  {X_test_tfidf.shape}")

TF-IDF Training Matrix Shape: (8, 54)
TF-IDF Testing Matrix Shape:  (2, 54)


In [7]:
# 1. Train Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, nb_preds)

# 2. Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, lr_preds)

# Print Benchmark Results
print("=" * 55)
print(f"Multinomial Naive Bayes Accuracy: {nb_acc * 100:.2f}%")
print("=" * 55)
print(classification_report(y_test, nb_preds, zero_division=0))

print("=" * 55)
print(f"Logistic Regression Accuracy:     {lr_acc * 100:.2f}%")
print("=" * 55)
print(classification_report(y_test, lr_preds, zero_division=0))

# Select best model for the prediction helper
best_model = lr_model if lr_acc >= nb_acc else nb_model
best_model_name = "Logistic Regression" if lr_acc >= nb_acc else "Multinomial Naive Bayes"
print(f"Active Model for Live Predictions: {best_model_name}")

Multinomial Naive Bayes Accuracy: 0.00%
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00       2.0
    positive       0.00      0.00      0.00       0.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0

Logistic Regression Accuracy:     0.00%
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00       2.0
    positive       0.00      0.00      0.00       0.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0

Active Model for Live Predictions: Logistic Regression


In [8]:
def predict_sentiment(review_text, model=best_model):
    # Preprocess the custom input using the exact same cleaning function
    cleaned = clean_text(review_text)
    # Transform into TF-IDF vector
    vectorized = tfidf.transform([cleaned])
    # Predict sentiment and confidence score
    prediction = model.predict(vectorized)[0]

    if hasattr(model, "predict_proba"):
        confidence = np.max(model.predict_proba(vectorized)) * 100
        return f"Sentiment: {prediction} (Confidence: {confidence:.1f}%)"
    return f"Sentiment: {prediction}"

# --- Test with sample custom phrases ---
sample_reviews = [
    "This product exceeded all my expectations! Super fast delivery and great build quality.",
    "Terrible experience. It broke after two days and customer support refused to help.",
    "The movie was okay, a bit slow in the middle but decent acting overall."
]

print("--- Sample Live Predictions ---")
for phrase in sample_reviews:
    print(f"Review: \"{phrase}\"\n -> {predict_sentiment(phrase)}\n")

# --- Interactive User Input Loop ---
print("--- Interactive Sentiment Tester (type 'quit' to stop) ---")
while True:
    user_input = input("Enter a review to classify: ")
    if user_input.strip().lower() in ['quit', 'exit', 'q']:
        print("Exiting interactive tester.")
        break
    if user_input.strip():
        print(f" -> {predict_sentiment(user_input)}\n")

--- Sample Live Predictions ---
Review: "This product exceeded all my expectations! Super fast delivery and great build quality."
 -> Sentiment: positive (Confidence: 70.4%)

Review: "Terrible experience. It broke after two days and customer support refused to help."
 -> Sentiment: positive (Confidence: 59.8%)

Review: "The movie was okay, a bit slow in the middle but decent acting overall."
 -> Sentiment: positive (Confidence: 55.2%)

--- Interactive Sentiment Tester (type 'quit' to stop) ---
Enter a review to classify: quit
Exiting interactive tester.
